<a href="https://colab.research.google.com/github/drmuruga/Tardigrade-CAHS-d-simulation/blob/main/tardigrade_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tardigrade CAHS D Protein Gelation Pipeline
This notebook creates the project structure, builds the memory-safe simulation modules, and executes the OpenMM run.

In [ ]:
# 1. Create the necessary folders
!mkdir -p src data/raw data/processed

# 2. Install the required libraries safely
!pip install openmm py3Dmol

In [ ]:
%%writefile src/system_builder.py
from openmm.app import PDBFile, ForceField, Modeller, PME, HBonds
from openmm.unit import nanometers

def build_system(pdb_path, padding_nm=0.8):
    print(f"Loading structure from {pdb_path}...")
    pdb = PDBFile(pdb_path)
    forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
    modeller = Modeller(pdb.topology, pdb.positions)

    print(f"Adding solvent box with {padding_nm} nm padding...")
    modeller.addSolvent(forcefield, padding=padding_nm * nanometers)

    system = forcefield.createSystem(
        modeller.topology,
        nonbondedMethod=PME,
        nonbondedCutoff=1.0 * nanometers,
        constraints=HBonds
    )
    return modeller.topology, modeller.positions, system

In [ ]:
%%writefile src/simulation.py
import gc
from openmm.app import Simulation, DCDReporter, StateDataReporter
from openmm import LangevinMiddleIntegrator
from openmm.unit import kelvin, picosecond, picoseconds

def run_simulation(topology, positions, system, output_dcd='data/processed/output.dcd', steps=5000, chunk_size=1000):
    integrator = LangevinMiddleIntegrator(300 * kelvin, 1 / picosecond, 0.004 * picoseconds)
    simulation = Simulation(topology, system, integrator)
    simulation.context.setPositions(positions)

    print("Minimizing energy...")
    simulation.minimizeEnergy()

    simulation.reporters.append(DCDReporter(output_dcd, chunk_size))
    simulation.reporters.append(StateDataReporter('data/processed/md_log.csv', chunk_size, step=True, potentialEnergy=True, temperature=True))

    print(f"Running {steps} steps in batches of {chunk_size}...")
    for i in range(0, steps, chunk_size):
        simulation.step(chunk_size)
        print(f"Completed step {i + chunk_size}")

        # Clear unused memory to prevent RAM crashing
        gc.collect()

    print("Simulation complete. Data safely saved to disk.")

In [ ]:
%%writefile src/visualization.py
import py3Dmol

def render_protein_segments(pdb_path):
    with open(pdb_path, 'r') as f:
        pdb_data = f.read()

    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_data, 'pdb')

    # Highlight chains for gelation architecture
    view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'blue'}})
    view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'orange'}})
    view.setStyle({'hetflag': True}, {'stick': {'radius': 0.3}})

    view.zoomTo()
    return view

### 🛑 UPLOAD YOUR DATA NOW 🛑
Before running the final cell below, open the left sidebar folder icon and drag `cahs_d_multichain.pdb` into the `data/raw/` folder!

In [ ]:
from src.system_builder import build_system
from src.simulation import run_simulation
from src.visualization import render_protein_segments

# 1. Visualize Initial State
print("Rendering initial structure...")
view = render_protein_segments('data/raw/cahs_d_multichain.pdb')
view.show()

# 2. Build System & Run Memory-Safe Simulation
top, pos, sys = build_system('data/raw/cahs_d_multichain.pdb', padding_nm=0.8)
run_simulation(top, pos, sys, steps=5000, chunk_size=1000)